# Grad-CAM: which part of the image decided the answer

A heatmap over the input showing where the evidence for a class came from — the most immediately useful debugging tool in this chapter.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 10 — Interpreting What ConvNets Learn](../../../course-web-slides/ch10/index.html) &nbsp;·&nbsp; **Section:** 03 — Visualizing heatmaps of class activation

---

## A pretrained classifier and a test image

In [ ]:
import keras
import numpy as np
import matplotlib.pyplot as plt

model = keras.applications.xception.Xception(weights="imagenet")

img_path = keras.utils.get_file(
    fname="elephant.jpg",
    origin="https://img-datasets.s3.amazonaws.com/elephant.jpg")

def get_img_array(img_path, target_size):
    img = keras.utils.load_img(img_path, target_size=target_size)
    array = keras.utils.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return keras.applications.xception.preprocess_input(array)

img_array = get_img_array(img_path, target_size=(299, 299))
preds = model.predict(img_array, verbose=0)
print(keras.applications.xception.decode_predictions(preds, top=3)[0])

Expected output:

```
[('n02504458', 'African_elephant', 0.87...),
 ('n01871265', 'tusker', 0.08...),
 ('n02504013', 'Indian_elephant', 0.02...)]
```

## The two pieces Grad-CAM needs

In [ ]:
last_conv_layer_name = "block14_sepconv2_act"
classifier_layer_names = ["avg_pool", "predictions"]

last_conv_layer = model.get_layer(last_conv_layer_name)
last_conv_layer_model = keras.Model(model.inputs, last_conv_layer.output)

classifier_input = keras.Input(shape=last_conv_layer.output.shape[1:])
x = classifier_input
for layer_name in classifier_layer_names:
    x = model.get_layer(layer_name)(x)
classifier_model = keras.Model(classifier_input, x)

The model is split at the **last convolutional layer**: the deepest place that still has spatial structure. Later than that and there is no *where* left to point at; earlier and the features are not class-specific enough to be informative.

## The gradient of the class score, per channel

In [ ]:
import tensorflow as tf

with tf.GradientTape() as tape:
    last_conv_layer_output = last_conv_layer_model(img_array)
    tape.watch(last_conv_layer_output)
    preds = classifier_model(last_conv_layer_output)
    top_pred_index = tf.argmax(preds[0])
    top_class_channel = preds[:, top_pred_index]

grads = tape.gradient(top_class_channel, last_conv_layer_output)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy()
print("one importance weight per channel:", pooled_grads.shape)

`pooled_grads[i]` answers: **how much does channel *i* matter to this class?** Weight each channel's activation map by that number and sum — the result is a map of where the evidence was.

## The heatmap

In [ ]:
last_conv_layer_output = last_conv_layer_output.numpy()[0]
for i in range(pooled_grads.shape[-1]):
    last_conv_layer_output[:, :, i] *= pooled_grads[i]
heatmap = np.mean(last_conv_layer_output, axis=-1)

heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)
plt.matshow(heatmap); plt.title("Raw heatmap (10x10)"); plt.show()

## Superimposed

In [ ]:
import matplotlib.cm as cm

img = keras.utils.load_img(img_path)
img = keras.utils.img_to_array(img)

hm = np.uint8(255 * heatmap)
jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[hm]

jet_heatmap = keras.utils.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = keras.utils.img_to_array(jet_heatmap)

superimposed = jet_heatmap * 0.4 + img
superimposed = keras.utils.array_to_img(superimposed)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.imshow(keras.utils.load_img(img_path)); a1.axis("off"); a1.set_title("input")
a2.imshow(superimposed); a2.axis("off"); a2.set_title("Grad-CAM: African elephant")
plt.tight_layout(); plt.show()

The heat concentrates on the **ears** — which is in fact the feature that separates African from Indian elephants. The model found the same discriminator a zoologist would name.

## The same heatmap for the second-place class

In [ ]:
def grad_cam(img_array, class_index):
    with tf.GradientTape() as tape:
        conv_out = last_conv_layer_model(img_array)
        tape.watch(conv_out)
        preds = classifier_model(conv_out)
        channel = preds[:, class_index]
    grads = tape.gradient(channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy()
    out = conv_out.numpy()[0]
    for i in range(pooled.shape[-1]):
        out[:, :, i] *= pooled[i]
    hm = np.mean(out, axis=-1)
    hm = np.maximum(hm, 0)
    return hm / (hm.max() + 1e-8)

preds = model.predict(img_array, verbose=0)
top3 = preds[0].argsort()[::-1][:3]
labels = keras.applications.xception.decode_predictions(preds, top=3)[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, idx, (_, name, score) in zip(axes, top3, labels):
    ax.matshow(grad_cam(img_array, idx))
    ax.set_title(f"{name}  {score:.2f}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

Different classes, different evidence — *tusker* looks at the tusks. **This is the check that catches a model doing the right thing for the wrong reason**: a classifier that finds cows by looking at grass will show you the grass.

## Use it on your own errors

The practical workflow: take the misclassified samples from your test set, run Grad-CAM on each, and look. In practice you will find one of three things — a genuinely hard image, a wrong label, or **the model attending to a background artifact**. Only the third is a modelling problem, and you cannot tell them apart from the confusion matrix.

---

## What to take away

- Grad-CAM weights each channel of the last convolutional layer by how much it moved the class score.
- Split the model at the **last layer with spatial structure**.
- Different classes produce different heatmaps — the ears against the tusks.
- Run it on your errors; it distinguishes a hard image from a model looking at the wrong thing.